# EDA Inicial — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Integrantes:** Juan Martínez Fraile · Otra persona la que sea
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** Análisis Exploratorio de Datos

---

## Objetivo

Realizar un análisis exploratorio del dataset de reservas hoteleras para:

1. Comprender la estructura y calidad de los datos.
2. Identificar problemas de modelado (data leakage, nulos, outliers, desbalanceo).
3. Justificar las decisiones de preprocesamiento y modelado de fases posteriores.
4. Seleccionar la métrica principal de evaluación.

## Temas a tratar

0. Configuración del notebook
1. Carga inicial y vista general de los datos
2. Análisis de la variable objetivo (`is_canceled`)
3. Estadísticas de variables numéricas
4. Estadísticas de variables categóricas
5. Análisis de nulos y valores especiales
6. Detección de data leakage
7. Análisis bivariado y correlaciones
8. Resumen de decisiones para la fase de preprocesamiento

## 0. Configuración del notebook

Importamos las librerías necesarias y definimos constantes/funciones que usaremos a lo largo del análisis.

### Importación de librerías

In [3]:
"""Importación de librerías necesarias para el análisis exploratorio."""

# Standard library
from pathlib import Path

# Análisis y manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Configuración global de visualización
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Librerías importadas correctamente.")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Librerías importadas correctamente.
pandas: 2.2.3
numpy: 1.26.4


### Definición de constantes

In [4]:
"""Constantes utilizadas a lo largo del análisis exploratorio."""

# === Rutas ===
# Path(__vsc_ipynb_file__) no existe; usamos la raíz del proyecto como referencia.
# Como el notebook está en notebooks/exploracion/, subimos dos niveles a la raíz.
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
PATH_DATA_RAW = PATH_PROYECTO / "data" / "raw"
PATH_DATA_PROCESSED = PATH_PROYECTO / "data" / "processed"
PATH_OUTPUTS = PATH_PROYECTO / "outputs"

PATH_DATASET = PATH_DATA_RAW / "dataset_practica_final.csv"

# === Configuración del análisis ===
SEED = 42  # Semilla para reproducibilidad
COLOR_PALETTE = ["#1f77b4", "#ff7f0e"]  # Azul (no cancelado) / Naranja (cancelado)
TARGET_COLUMN = "is_canceled"

# Verificación visual
print(f"Raíz del proyecto:  {PATH_PROYECTO}")
print(f"Dataset:            {PATH_DATASET}")
print(f"¿Dataset existe?:   {PATH_DATASET.exists()}")
print(f"Tamaño del archivo: {PATH_DATASET.stat().st_size / 1024 / 1024:.2f} MB")

Raíz del proyecto:  c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
Dataset:            c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml\data\raw\dataset_practica_final.csv
¿Dataset existe?:   True
Tamaño del archivo: 16.07 MB


### Definición de funciones

In [5]:
"""Funciones de ayuda reutilizables en todo el análisis exploratorio."""

from typing import Optional


def resumen_df(df: pd.DataFrame) -> pd.DataFrame:
    """Devuelve un resumen compacto de cada columna de un DataFrame.

    Para cada columna calcula tipo, número de valores únicos, número y porcentaje
    de nulos, y una muestra de valores. Es la primera vista útil tras cargar
    cualquier dataset nuevo.

    Args:
        df (pd.DataFrame): DataFrame a inspeccionar.

    Returns:
        pd.DataFrame: Tabla con una fila por columna del DataFrame original.

    Example:
        >>> resumen_df(df).head()
    """
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_unicos": df.nunique(),
        "n_nulos": df.isnull().sum(),
        "pct_nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "ejemplos": [df[col].dropna().unique()[:3].tolist() for col in df.columns],
    })
    return resumen.sort_values("pct_nulos", ascending=False)


def plot_distribucion_categorica(
    df: pd.DataFrame,
    columna: str,
    hue: Optional[str] = None,
    top_n: Optional[int] = None,
    figsize: tuple = (10, 5),
    title: Optional[str] = None,
) -> None:
    """Visualiza la distribución de una variable categórica con conteos.

    Útil para entender la composición de variables como `hotel`, `market_segment`
    o `country`. Si se pasa `hue`, descompone cada barra por la variable indicada
    (típicamente la variable objetivo).

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        columna (str): Nombre de la columna categórica a visualizar.
        hue (Optional[str]): Variable secundaria para descomponer (ej. el target).
        top_n (Optional[int]): Si se indica, muestra solo las `top_n` categorías más frecuentes.
        figsize (tuple): Tamaño de la figura (ancho, alto).
        title (Optional[str]): Título personalizado. Por defecto se genera automáticamente.

    Returns:
        None: La función dibuja directamente con matplotlib/seaborn.
    """
    plt.figure(figsize=figsize)

    # Filtrar a top_n si aplica
    if top_n is not None:
        top_categorias = df[columna].value_counts().head(top_n).index
        df_plot = df[df[columna].isin(top_categorias)]
        orden = top_categorias
    else:
        df_plot = df
        orden = df[columna].value_counts().index

    sns.countplot(
        data=df_plot,
        y=columna,
        hue=hue,
        order=orden,
        palette=COLOR_PALETTE if hue == TARGET_COLUMN else "deep",
    )

    if title is None:
        title = f"Distribución de '{columna}'"
        if hue:
            title += f" por '{hue}'"
        if top_n:
            title += f" (top {top_n})"

    plt.title(title)
    plt.xlabel("Número de reservas")
    plt.tight_layout()
    plt.show()


def plot_distribucion_numerica(
    df: pd.DataFrame,
    columna: str,
    hue: Optional[str] = None,
    bins: int = 50,
    figsize: tuple = (12, 4),
) -> None:
    """Visualiza histograma + boxplot de una variable numérica.

    Combina dos vistas complementarias: el histograma muestra la forma de la distribución,
    el boxplot muestra mediana, cuartiles y outliers. Si se pasa `hue`, separa por clases.

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        columna (str): Nombre de la columna numérica.
        hue (Optional[str]): Variable categórica para separar las distribuciones.
        bins (int): Número de bins del histograma.
        figsize (tuple): Tamaño de la figura.

    Returns:
        None
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Histograma
    sns.histplot(
        data=df,
        x=columna,
        hue=hue,
        bins=bins,
        ax=axes[0],
        palette=COLOR_PALETTE if hue == TARGET_COLUMN else None,
        kde=True,
    )
    axes[0].set_title(f"Histograma de '{columna}'")

    # Boxplot
    if hue:
        sns.boxplot(
            data=df,
            x=hue,
            y=columna,
            ax=axes[1],
            palette=COLOR_PALETTE if hue == TARGET_COLUMN else None,
        )
        axes[1].set_title(f"Boxplot de '{columna}' por '{hue}'")
    else:
        sns.boxplot(data=df, x=columna, ax=axes[1])
        axes[1].set_title(f"Boxplot de '{columna}'")

    plt.tight_layout()
    plt.show()


print("Funciones helper definidas correctamente.")

Funciones helper definidas correctamente.


## 1. Carga inicial y vista general de los datos

En esta sección cargamos el dataset desde `data/raw/` y obtenemos una **primera fotografía** de su estructura:

- Dimensiones (filas, columnas).
- Tipos de datos por columna.
- Muestras de las primeras y últimas filas.
- Estadísticas descriptivas básicas.
- Detección de duplicados.

El objetivo es **no tomar decisiones todavía**, solo observar.

### 1.1. Lectura del dataset

In [6]:
"""Carga del dataset desde data/raw/."""

df = pd.read_csv(PATH_DATASET)

print(f"Dataset cargado: {PATH_DATASET.name}")
print(f"Dimensiones:      {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria ocupada:  {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

Dataset cargado: dataset_practica_final.csv
Dimensiones:      119,390 filas × 32 columnas
Memoria ocupada:  104.83 MB


### 1.2. Vista previa de los datos

Inspeccionamos las primeras y últimas filas para entender la "pinta" de las columnas:
qué tipos de valores contienen, si hay patrones visibles, qué columnas parecen
identificadores y cuáles parecen claramente numéricas o categóricas.

In [7]:
"""Primeras filas del DataFrame."""

df.head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,NaN,NaN,0,Transient,107.0,0,0,Check-Out,2015-07-03
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,303.0,NaN,0,Transient,103.0,0,1,Check-Out,2015-07-03
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,82.0,0,1,Canceled,2015-05-06
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,0.0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,D,D,0,No Deposit,15.0,NaN,0,Transient,105.5,0,0,Canceled,2015-04-22


In [8]:
"""Últimas filas del DataFrame."""

df.tail(5)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
119385,City Hotel,0,23,2017,August,35,30,2,5,2,0.0,0,BB,BEL,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,0.0,0,BB,FRA,Online TA,TA/TO,0,0,0,E,E,0,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,0.0,0,BB,DEU,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07
119389,City Hotel,0,205,2017,August,35,29,2,7,2,0.0,0,HB,DEU,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,151.20,0,2,Check-Out,2017-09-07


### 1.3. Tipos de datos y estructura

Inspeccionamos los tipos de cada columna. Buscamos:

- Columnas numéricas (int/float).
- Columnas categóricas (object).
- Columnas que parecen numéricas pero podrían ser categóricas disfrazadas (`agent`, `company`, `is_repeated_guest`).
- Columnas con tipo incorrecto (ej. una fecha guardada como string).

In [9]:
"""Información de tipos y nulos por columna."""

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [10]:
"""Resumen compacto con ejemplos por columna."""

resumen_df(df)

,tipo,n_unicos,n_nulos,pct_nulos,ejemplos
company,float64,352,112593,94.31,"[110.0, 113.0, 270.0]"
agent,float64,333,16340,13.69,"[304.0, 240.0, 303.0]"
country,object,177,488,0.41,"[PRT, GBR, USA]"
hotel,object,2,0,0.00,"[Resort Hotel, City Hotel]"
previous_cancellations,int64,15,0,0.00,"[0, 1, 2]"
reservation_status,object,3,0,0.00,"[Check-Out, Canceled, No-Show]"
total_of_special_requests,int64,6,0,0.00,"[0, 1, 3]"
required_car_parking_spaces,int64,5,0,0.00,"[0, 1, 2]"
adr,float64,8879,0,0.00,"[0.0, 75.0, 98.0]"
customer_type,object,4,0,0.00,"[Transient, Contract, Transient-Party]"


### 1.4. Estadísticas descriptivas

Para las variables numéricas observamos rango, media, mediana, mínimos y máximos.
Esto nos permite detectar:

- Valores imposibles (ej. una reserva con 0 adultos, 0 niños y 0 bebés).
- Outliers extremos (ej. `lead_time` de 700 días o más).
- Escalas muy diferentes entre variables (importante para futuros modelos con regularización).

In [11]:
"""Estadísticas descriptivas de las variables numéricas."""

df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.37,0.48,0.00,0.00,0.00,1.0,1.0
lead_time,119390.0,104.01,106.86,0.00,18.00,69.00,160.0,737.0
arrival_date_year,119390.0,2016.16,0.71,2015.00,2016.00,2016.00,2017.0,2017.0
arrival_date_week_number,119390.0,27.17,13.61,1.00,16.00,28.00,38.0,53.0
arrival_date_day_of_month,119390.0,15.80,8.78,1.00,8.00,16.00,23.0,31.0
stays_in_weekend_nights,119390.0,0.93,1.00,0.00,0.00,1.00,2.0,19.0
stays_in_week_nights,119390.0,2.50,1.91,0.00,1.00,2.00,3.0,50.0
adults,119390.0,1.86,0.58,0.00,2.00,2.00,2.0,55.0
children,119386.0,0.10,0.40,0.00,0.00,0.00,0.0,10.0
babies,119390.0,0.01,0.10,0.00,0.00,0.00,0.0,10.0


### 1.5. Detección de duplicados

Comprobamos si hay filas exactamente duplicadas. En reservas hoteleras esto puede pasar
por errores de captura o por reservas auténticamente repetidas (mismo cliente, mismas
fechas). Si hay muchos duplicados deberíamos analizarlos antes de tirarlos.

In [12]:
"""Detección de filas duplicadas."""

n_duplicados = df.duplicated().sum()
pct_duplicados = (n_duplicados / len(df)) * 100

print(f"Filas duplicadas: {n_duplicados:,} ({pct_duplicados:.2f}%)")

Filas duplicadas: 31,994 (26.80%)


In [13]:
"""Investigación de las filas duplicadas."""

# Coger las primeras 6 filas marcadas como duplicadas
duplicados_ejemplo = df[df.duplicated(keep=False)].head(6)
duplicados_ejemplo

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
21,Resort Hotel,0,72,2015,July,27,1,2,4,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
39,Resort Hotel,0,70,2015,July,27,2,2,3,2,0.0,0,HB,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
43,Resort Hotel,0,70,2015,July,27,2,2,3,2,0.0,0,HB,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07


### 1.6. Resumen de la Sección 1

**Lo que hemos descubierto:**

- Dataset: **119.390 filas × 32 columnas**, cubre **3 años** (2015–2017).
- Tipos: ~16 numéricas + ~12 categóricas + ~4 floats.
- **Nulos**: `company` (~94%, inservible), `agent` (~13.7%), `country` (~0.4%), `children` (despreciable).
- **Valores anómalos** detectados:
  - `adr` puede ser negativo y llegar a 5400 (outliers extremos).
  - `adults` con max 55 (probable error de captura).
  - Hay duplicados (~25%) pero son **reservas legítimamente repetidas**, no las eliminaremos.
- Variable objetivo: `is_canceled`, binaria. Analizada en detalle en la sección 2.

Pendiente para la Sección 2: estudiar a fondo `is_canceled` (balance de clases, distribución temporal, métrica de evaluación).